In [8]:
!uv pip install ollama

Using Python 3.12.0 environment at: D:\code\Collection_of_virtual\SLM_space
Resolved 12 packages in 900ms
 Downloaded pydantic-core
Prepared 5 packages in 612ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 5 packages in 205ms
 + annotated-types==0.7.0
 + ollama==0.6.1
 + pydantic==2.12.5
 + pydantic-core==2.41.5
 + typing-inspection==0.4.2


In [2]:
import time
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

def clear_mem():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

def get_vram_usage():
    return torch.cuda.memory_allocated() / 1024**2

def get_vram_max():
    return torch.cuda.max_memory_allocated() / 1024**2

#양자화
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # 님이 쓰려고 했던 설정
    bnb_4bit_compute_dtype=torch.float16, # 연산 속도를 위해 권장
    bnb_4bit_use_double_quant=True       # 2060 VRAM 절약을 위해 권장
)

cache_dir = r"D:\code\F1_Team_Radio\SLM_mobel"
prompt = "당신은 예의 바른 안내원입니다. AI의 장점을 한 문장으로 말해주세요."

d:\code\Collection_of_virtual\SLM_space\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 양자화 전
---

#####  Qwen2.5-0.5B

In [4]:
num_iterations = 10
latencies = []
tps_list = []

#----------모델 로드----------------------------
clear_mem()
initial_vram = get_vram_usage()


model_id = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_dir)
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    device_map="auto", 
    torch_dtype=torch.float16,
    cache_dir=cache_dir
)
model_loaded_vram = get_vram_usage()
actual_model_size = model_loaded_vram - initial_vram
#---------------------------------------------------------------

inputs = tokenizer(f"System: {prompt}\nUser: AI의 장점은?\nAssistant:", return_tensors="pt").to(model.device)
for _ in range(2):
    _ = model.generate(**inputs, max_new_tokens=64)

torch.cuda.reset_peak_memory_stats()

for i in range(num_iterations):
    start_time = time.time()
    outputs = model.generate(**inputs, max_new_tokens=64)
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    end_time = time.time()
    
    generated_tokens = outputs.shape[1] - inputs['input_ids'].shape[1]
    iter_latency = end_time - start_time
    tps = generated_tokens / iter_latency
    
    latencies.append(iter_latency)
    tps_list.append(tps)

peak_vram = get_vram_max()

avg_latency = sum(latencies) / num_iterations
min_latency = min(latencies)
max_latency = max(latencies)
avg_tps = sum(tps_list) / num_iterations

print("\n" + "="*40)
print(f"Report ({model_id})")
print(f"Avg Time: {avg_latency:.4f}s")
print(f"Min Time: {min_latency:.4f}s")
print(f"Max Time: {max_latency:.4f}s")
print(f"Avg Tokens/Sec: {avg_tps:.2f} t/s")
print(f"Model VRAM: {actual_model_size:.2f} MB")
print(f"Peak VRAM: {peak_vram:.2f} MB")
print("="*40)

del model, tokenizer
clear_mem()

Loading weights: 100%|██████████| 290/290 [00:23<00:00, 12.13it/s]



Report (Qwen/Qwen2.5-0.5B-Instruct)
Avg Time: 2.8749s
Min Time: 2.8501s
Max Time: 2.9001s
Avg Tokens/Sec: 22.26 t/s
Model VRAM: 942.82 MB
Peak VRAM: 959.42 MB


##### Qwen2.5-1.5B

In [5]:
num_iterations = 10
latencies = []
tps_list = []

#----------모델 로드----------------------------
clear_mem()
initial_vram = get_vram_usage()


model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_dir)
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    device_map="auto", 
    torch_dtype=torch.float16,
    cache_dir=cache_dir
)
model_loaded_vram = get_vram_usage()
actual_model_size = model_loaded_vram - initial_vram
#---------------------------------------------------------------

inputs = tokenizer(f"System: {prompt}\nUser: AI의 장점은?\nAssistant:", return_tensors="pt").to(model.device)
for _ in range(2):
    _ = model.generate(**inputs, max_new_tokens=64)

torch.cuda.reset_peak_memory_stats()

for i in range(num_iterations):
    start_time = time.time()
    outputs = model.generate(**inputs, max_new_tokens=64)
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    end_time = time.time()
    
    generated_tokens = outputs.shape[1] - inputs['input_ids'].shape[1]
    iter_latency = end_time - start_time
    tps = generated_tokens / iter_latency
    
    latencies.append(iter_latency)
    tps_list.append(tps)

peak_vram = get_vram_max()

avg_latency = sum(latencies) / num_iterations
min_latency = min(latencies)
max_latency = max(latencies)
avg_tps = sum(tps_list) / num_iterations

print("\n" + "="*40)
print(f"Report ({model_id})")
print(f"Avg Time: {avg_latency:.4f}s")
print(f"Min Time: {min_latency:.4f}s")
print(f"Max Time: {max_latency:.4f}s")
print(f"Avg Tokens/Sec: {avg_tps:.2f} t/s")
print(f"Model VRAM: {actual_model_size:.2f} MB")
print(f"Peak VRAM: {peak_vram:.2f} MB")
print("="*40)

del model, tokenizer
clear_mem()

Loading weights: 100%|██████████| 338/338 [01:43<00:00,  3.26it/s] 



Report (Qwen/Qwen2.5-1.5B-Instruct)
Avg Time: 3.3213s
Min Time: 3.2800s
Max Time: 3.3951s
Avg Tokens/Sec: 19.27 t/s
Model VRAM: 2944.40 MB
Peak VRAM: 2963.08 MB


##### gemma-2-2b

In [6]:
num_iterations = 10
latencies = []
tps_list = []

#----------모델 로드----------------------------
clear_mem()
initial_vram = get_vram_usage()


model_id = "google/gemma-2-2b-it"

tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_dir)
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    device_map="auto", 
    torch_dtype=torch.float16,
    cache_dir=cache_dir
)
model_loaded_vram = get_vram_usage()
actual_model_size = model_loaded_vram - initial_vram
#---------------------------------------------------------------

inputs = tokenizer(f"System: {prompt}\nUser: AI의 장점은?\nAssistant:", return_tensors="pt").to(model.device)
for _ in range(2):
    _ = model.generate(**inputs, max_new_tokens=64)

torch.cuda.reset_peak_memory_stats()

for i in range(num_iterations):
    start_time = time.time()
    outputs = model.generate(**inputs, max_new_tokens=64)
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    end_time = time.time()
    
    generated_tokens = outputs.shape[1] - inputs['input_ids'].shape[1]
    iter_latency = end_time - start_time
    tps = generated_tokens / iter_latency
    
    latencies.append(iter_latency)
    tps_list.append(tps)

peak_vram = get_vram_max()

avg_latency = sum(latencies) / num_iterations
min_latency = min(latencies)
max_latency = max(latencies)
avg_tps = sum(tps_list) / num_iterations

print("\n" + "="*40)
print(f"Report ({model_id})")
print(f"Avg Time: {avg_latency:.4f}s")
print(f"Min Time: {min_latency:.4f}s")
print(f"Max Time: {max_latency:.4f}s")
print(f"Avg Tokens/Sec: {avg_tps:.2f} t/s")
print(f"Model VRAM: {actual_model_size:.2f} MB")
print(f"Peak VRAM: {peak_vram:.2f} MB")
print("="*40)

del model, tokenizer
clear_mem()

Loading weights: 100%|██████████| 288/288 [02:19<00:00,  2.07it/s]  
Some parameters are on the meta device because they were offloaded to the cpu.



Report (google/gemma-2-2b-it)
Avg Time: 8.4399s
Min Time: 7.7501s
Max Time: 9.4783s
Avg Tokens/Sec: 4.28 t/s
Model VRAM: 4392.39 MB
Peak VRAM: 4449.68 MB


##### gemma-2-2b-abliterated

In [3]:
num_iterations = 10
latencies = []
tps_list = []

#----------모델 로드----------------------------
clear_mem()
initial_vram = get_vram_usage()


model_id = "IlyaGusev/gemma-2-2b-it-abliterated"

tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_dir)
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    device_map="auto", 
    torch_dtype=torch.float16,
    cache_dir=cache_dir
)
model_loaded_vram = get_vram_usage()
actual_model_size = model_loaded_vram - initial_vram
#---------------------------------------------------------------

inputs = tokenizer(f"System: {prompt}\nUser: AI의 장점은?\nAssistant:", return_tensors="pt").to(model.device)
for _ in range(2):
    _ = model.generate(**inputs, max_new_tokens=64)

torch.cuda.reset_peak_memory_stats()

for i in range(num_iterations):
    start_time = time.time()
    outputs = model.generate(**inputs, max_new_tokens=64)
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    end_time = time.time()
    
    generated_tokens = outputs.shape[1] - inputs['input_ids'].shape[1]
    iter_latency = end_time - start_time
    tps = generated_tokens / iter_latency
    
    latencies.append(iter_latency)
    tps_list.append(tps)

peak_vram = get_vram_max()

avg_latency = sum(latencies) / num_iterations
min_latency = min(latencies)
max_latency = max(latencies)
avg_tps = sum(tps_list) / num_iterations

print("\n" + "="*40)
print(f"Report ({model_id})")
print(f"Avg Time: {avg_latency:.4f}s")
print(f"Min Time: {min_latency:.4f}s")
print(f"Max Time: {max_latency:.4f}s")
print(f"Avg Tokens/Sec: {avg_tps:.2f} t/s")
print(f"Model VRAM: {actual_model_size:.2f} MB")
print(f"Peak VRAM: {peak_vram:.2f} MB")
print("="*40)

del model, tokenizer
clear_mem()

Loading weights: 100%|██████████| 288/288 [00:35<00:00,  8.07it/s]
Some parameters are on the meta device because they were offloaded to the cpu.



Report (IlyaGusev/gemma-2-2b-it-abliterated)
Avg Time: 11.0944s
Min Time: 10.1685s
Max Time: 11.8484s
Avg Tokens/Sec: 3.98 t/s
Model VRAM: 4392.39 MB
Peak VRAM: 4450.49 MB


##### Llama-3.1-8B

In [ ]:
#2060 실행 불가
num_iterations = 10
latencies = []
tps_list = []

#----------모델 로드----------------------------
clear_mem()
initial_vram = get_vram_usage()


model_id = "meta-llama/Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_dir)
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    device_map="auto", 
    torch_dtype=torch.float16,
    cache_dir=cache_dir
)
model_loaded_vram = get_vram_usage()
actual_model_size = model_loaded_vram - initial_vram
#---------------------------------------------------------------

inputs = tokenizer(f"System: {prompt}\nUser: AI의 장점은?\nAssistant:", return_tensors="pt").to(model.device)
for _ in range(2):
    _ = model.generate(**inputs, max_new_tokens=64)

torch.cuda.reset_peak_memory_stats()

for i in range(num_iterations):
    start_time = time.time()
    outputs = model.generate(**inputs, max_new_tokens=64)
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    end_time = time.time()
    
    generated_tokens = outputs.shape[1] - inputs['input_ids'].shape[1]
    iter_latency = end_time - start_time
    tps = generated_tokens / iter_latency
    
    latencies.append(iter_latency)
    tps_list.append(tps)

peak_vram = get_vram_max()

avg_latency = sum(latencies) / num_iterations
min_latency = min(latencies)
max_latency = max(latencies)
avg_tps = sum(tps_list) / num_iterations

print("\n" + "="*40)
print(f"Report ({model_id})")
print(f"Avg Time: {avg_latency:.4f}s")
print(f"Min Time: {min_latency:.4f}s")
print(f"Max Time: {max_latency:.4f}s")
print(f"Avg Tokens/Sec: {avg_tps:.2f} t/s")
print(f"Model VRAM: {actual_model_size:.2f} MB")
print(f"Peak VRAM: {peak_vram:.2f} MB")
print("="*40)

del model, tokenizer
clear_mem()

# 양자화 후

#### Qwen2.5-0.5B

In [12]:
num_iterations = 10
latencies = []
tps_list = []

#----------모델 로드----------------------------
clear_mem()
initial_vram = get_vram_usage()


model_id = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_dir)
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    device_map="auto", 
    torch_dtype=torch.float16,
    quantization_config=bnb_config,
    cache_dir=cache_dir
)
model_loaded_vram = get_vram_usage()
actual_model_size = model_loaded_vram - initial_vram
#---------------------------------------------------------------

inputs = tokenizer(f"System: {prompt}\nUser: AI의 장점은?\nAssistant:", return_tensors="pt").to(model.device)
for _ in range(2):
    _ = model.generate(**inputs, max_new_tokens=64)

torch.cuda.reset_peak_memory_stats()

for i in range(num_iterations):
    start_time = time.time()
    outputs = model.generate(**inputs, max_new_tokens=64)
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    end_time = time.time()
    
    generated_tokens = outputs.shape[1] - inputs['input_ids'].shape[1]
    iter_latency = end_time - start_time
    tps = generated_tokens / iter_latency
    
    latencies.append(iter_latency)
    tps_list.append(tps)

peak_vram = get_vram_max()

avg_latency = sum(latencies) / num_iterations
min_latency = min(latencies)
max_latency = max(latencies)
avg_tps = sum(tps_list) / num_iterations

print("\n" + "="*40)
print(f"Report ({model_id})")
print(f"Avg Time: {avg_latency:.4f}s")
print(f"Min Time: {min_latency:.4f}s")
print(f"Max Time: {max_latency:.4f}s")
print(f"Avg Tokens/Sec: {avg_tps:.2f} t/s")
print(f"Model VRAM: {actual_model_size:.2f} MB")
print(f"Peak VRAM: {peak_vram:.2f} MB")
print("="*40)

del model, tokenizer
clear_mem()

Loading weights: 100%|██████████| 290/290 [00:25<00:00, 11.22it/s]



Report (Qwen/Qwen2.5-0.5B-Instruct)
Avg Time: 5.0957s
Min Time: 4.8004s
Max Time: 5.3230s
Avg Tokens/Sec: 12.57 t/s
Model VRAM: 437.58 MB
Peak VRAM: 455.35 MB


#### Qwen2.5-1.5B

In [ ]:
num_iterations = 10
latencies = []
tps_list = []

#----------모델 로드----------------------------
clear_mem()
initial_vram = get_vram_usage()


model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_dir)
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    device_map="auto", 
    torch_dtype=torch.float16,
    quantization_config=bnb_config,
    cache_dir=cache_dir
)
model_loaded_vram = get_vram_usage()
actual_model_size = model_loaded_vram - initial_vram
#---------------------------------------------------------------

inputs = tokenizer(f"System: {prompt}\nUser: AI의 장점은?\nAssistant:", return_tensors="pt").to(model.device)
for _ in range(2):
    _ = model.generate(**inputs, max_new_tokens=64)

torch.cuda.reset_peak_memory_stats()

for i in range(num_iterations):
    start_time = time.time()
    outputs = model.generate(**inputs, max_new_tokens=64)
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    end_time = time.time()
    
    generated_tokens = outputs.shape[1] - inputs['input_ids'].shape[1]
    iter_latency = end_time - start_time
    tps = generated_tokens / iter_latency
    
    latencies.append(iter_latency)
    tps_list.append(tps)

peak_vram = get_vram_max()

avg_latency = sum(latencies) / num_iterations
min_latency = min(latencies)
max_latency = max(latencies)
avg_tps = sum(tps_list) / num_iterations

print("\n" + "="*40)
print(f"Report ({model_id})")
print(f"Avg Time: {avg_latency:.4f}s")
print(f"Min Time: {min_latency:.4f}s")
print(f"Max Time: {max_latency:.4f}s")
print(f"Avg Tokens/Sec: {avg_tps:.2f} t/s")
print(f"Model VRAM: {actual_model_size:.2f} MB")
print(f"Peak VRAM: {peak_vram:.2f} MB")
print("="*40)

del model, tokenizer
clear_mem()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

#### gemma-2-2b

In [14]:
num_iterations = 10
latencies = []
tps_list = []

#----------모델 로드----------------------------
clear_mem()
initial_vram = get_vram_usage()


model_id = "google/gemma-2-2b-it"

tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_dir)
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    device_map="auto", 
    quantization_config=bnb_config,
    cache_dir=cache_dir
)
model_loaded_vram = get_vram_usage()
actual_model_size = model_loaded_vram - initial_vram
#---------------------------------------------------------------

inputs = tokenizer(f"System: {prompt}\nUser: AI의 장점은?\nAssistant:", return_tensors="pt").to(model.device)
for _ in range(2):
    _ = model.generate(**inputs, max_new_tokens=64)

torch.cuda.reset_peak_memory_stats()

for i in range(num_iterations):
    start_time = time.time()
    outputs = model.generate(**inputs, max_new_tokens=64)
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    end_time = time.time()
    
    generated_tokens = outputs.shape[1] - inputs['input_ids'].shape[1]
    iter_latency = end_time - start_time
    tps = generated_tokens / iter_latency
    
    latencies.append(iter_latency)
    tps_list.append(tps)

peak_vram = get_vram_max()

avg_latency = sum(latencies) / num_iterations
min_latency = min(latencies)
max_latency = max(latencies)
avg_tps = sum(tps_list) / num_iterations

print("\n" + "="*40)
print(f"Report ({model_id})")
print(f"Avg Time: {avg_latency:.4f}s")
print(f"Min Time: {min_latency:.4f}s")
print(f"Max Time: {max_latency:.4f}s")
print(f"Avg Tokens/Sec: {avg_tps:.2f} t/s")
print(f"Model VRAM: {actual_model_size:.2f} MB")
print(f"Peak VRAM: {peak_vram:.2f} MB")
print("="*40)

del model, tokenizer
clear_mem()

Loading weights: 100%|██████████| 288/288 [00:31<00:00,  9.18it/s]



Report (google/gemma-2-2b-it)
Avg Time: 4.8743s
Min Time: 4.7346s
Max Time: 5.4112s
Avg Tokens/Sec: 9.45 t/s
Model VRAM: 2145.42 MB
Peak VRAM: 2240.77 MB


##### gemma-2-2b-it-abliterated

In [3]:
num_iterations = 10
latencies = []
tps_list = []

#----------모델 로드----------------------------
clear_mem()
initial_vram = get_vram_usage()


model_id = "IlyaGusev/gemma-2-2b-it-abliterated"

tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_dir)
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    device_map="auto", 
    quantization_config=bnb_config,
    cache_dir=cache_dir
)
model_loaded_vram = get_vram_usage()
actual_model_size = model_loaded_vram - initial_vram
#---------------------------------------------------------------

inputs = tokenizer(f"System: {prompt}\nUser: AI의 장점은?\nAssistant:", return_tensors="pt").to(model.device)
for _ in range(2):
    _ = model.generate(**inputs, max_new_tokens=64)

torch.cuda.reset_peak_memory_stats()

for i in range(num_iterations):
    start_time = time.time()
    outputs = model.generate(**inputs, max_new_tokens=64)
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    end_time = time.time()
    
    generated_tokens = outputs.shape[1] - inputs['input_ids'].shape[1]
    iter_latency = end_time - start_time
    tps = generated_tokens / iter_latency
    
    latencies.append(iter_latency)
    tps_list.append(tps)

peak_vram = get_vram_max()

avg_latency = sum(latencies) / num_iterations
min_latency = min(latencies)
max_latency = max(latencies)
avg_tps = sum(tps_list) / num_iterations

print("\n" + "="*40)
print(f"Report ({model_id})")
print(f"Avg Time: {avg_latency:.4f}s")
print(f"Min Time: {min_latency:.4f}s")
print(f"Max Time: {max_latency:.4f}s")
print(f"Avg Tokens/Sec: {avg_tps:.2f} t/s")
print(f"Model VRAM: {actual_model_size:.2f} MB")
print(f"Peak VRAM: {peak_vram:.2f} MB")
print("="*40)

del model, tokenizer
clear_mem()

Loading weights: 100%|██████████| 288/288 [00:21<00:00, 13.65it/s]



Report (IlyaGusev/gemma-2-2b-it-abliterated)
Avg Time: 7.0090s
Min Time: 6.6711s
Max Time: 7.1259s
Avg Tokens/Sec: 8.13 t/s
Model VRAM: 2149.51 MB
Peak VRAM: 2245.74 MB


# ollama

In [4]:
import ollama
import time
def clear_vram(model_name):
    try:
        ollama.chat(model=model_name, keep_alive=0)
        print("VRAM 정리")
    except Exception as e:
        print(f"언로드 중 오류 발생: {e}")

def get_vram(model_name):
    try:
        ps_info = ollama.ps()
        for m in ps_info.get('models', []):
            if m.get('name') == model_name:
                # size_vram은 바이트 단위이므로 1024**2 로 나누어 MB로 변환
                model_vram_mb = m.get('size_vram', 0) / (1024 ** 2)
                break
    except Exception as e:
        print("VRAM 정보를 가져오는 데 실패했습니다:", e)

##### gemma2_GGUF

In [5]:
model_name = 'gemma2:2b'

# 워밍업 (모델을 VRAM에 확실히 로드)
ollama.chat(model=model_name, messages=[{'role': 'user', 'content': 'hi'}])

# Ollama ps API를 통해 VRAM 사용량 가져오기 (MB 단위 변환)
model_vram_mb = 0
get_vram(model_name)

latencies = []
tokens_per_second_list = []

for i in range(10):
    start_time = time.time()
    
    # options 파라미터를 사용해 max_new_tokens=64 와 동일한 효과 적용
    response = ollama.chat(model=model_name, messages=[
        {'role': 'user', 'content': 'AI의 장점은?'},
    ], options={
        'num_predict': 64  # 최대 생성 토큰 64개로 제한
    })
    
    end_time = time.time()
    iter_latency = end_time - start_time
    
    # Ollama가 자체적으로 계산한 순수 평가 시간 및 토큰 수 가져오기 (통신 딜레이 제외)
    eval_count = response.get('eval_count', 0) # 생성된 토큰 수
    eval_duration_ns = response.get('eval_duration', 0) # 생성에 걸린 나노초
    
    if eval_duration_ns > 0:
        tps = eval_count / (eval_duration_ns / 1e9) # 초당 토큰 생성 수 (TPS)
    else:
        tps = 0
        
    latencies.append(iter_latency)
    tokens_per_second_list.append(tps)
    
    # 진행 상황이 궁금하시면 아래 주석을 해제하세요.
    # print(f"[{i+1}/10] 소요 시간: {iter_latency:.2f}초 | 속도: {tps:.2f} t/s")

# 통계 계산
avg_latency = sum(latencies) / 10
min_latency = min(latencies)
max_latency = max(latencies)
avg_tps = sum(tokens_per_second_list) / 10

# 기존 실험 양식과 100% 동일하게 출력
print("\n" + "="*40)
print(f"Report ({model_name})")
print(f"Avg Time: {avg_latency:.4f}s")
print(f"Min Time: {min_latency:.4f}s")
print(f"Max Time: {max_latency:.4f}s")
print(f"Avg Tokens/Sec: {avg_tps:.2f} t/s")
print(f"Model VRAM: {model_vram_mb:.2f} MB")
print(f"Peak VRAM: {model_vram_mb:.2f} MB")
print("="*40)

clear_vram(model_name)


Report (gemma2:2b)
Avg Time: 0.8222s
Min Time: 0.8163s
Max Time: 0.8272s
Avg Tokens/Sec: 110.55 t/s
Model VRAM: 0.00 MB
Peak VRAM: 0.00 MB
VRAM 정리


##### Qwen2.5-7B_GGUF

In [ ]:
model_name = 'qwen2.5-coder:7b'

# 워밍업 (모델을 VRAM에 확실히 로드)
ollama.chat(model=model_name, messages=[{'role': 'user', 'content': 'hi'}])

# Ollama ps API를 통해 VRAM 사용량 가져오기 (MB 단위 변환)
model_vram_mb = 0
get_vram(model_name)

latencies = []
tokens_per_second_list = []

for i in range(10):
    start_time = time.time()
    
    # options 파라미터를 사용해 max_new_tokens=64 와 동일한 효과 적용
    response = ollama.chat(model=model_name, messages=[
        {'role': 'user', 'content': 'AI의 장점은?'},
    ], options={
        'num_predict': 64  # 최대 생성 토큰 64개로 제한
    })
    
    end_time = time.time()
    iter_latency = end_time - start_time
    
    # Ollama가 자체적으로 계산한 순수 평가 시간 및 토큰 수 가져오기 (통신 딜레이 제외)
    eval_count = response.get('eval_count', 0) # 생성된 토큰 수
    eval_duration_ns = response.get('eval_duration', 0) # 생성에 걸린 나노초
    
    if eval_duration_ns > 0:
        tps = eval_count / (eval_duration_ns / 1e9) # 초당 토큰 생성 수 (TPS)
    else:
        tps = 0
        
    latencies.append(iter_latency)
    tokens_per_second_list.append(tps)
    
    # 진행 상황이 궁금하시면 아래 주석을 해제하세요.
    # print(f"[{i+1}/10] 소요 시간: {iter_latency:.2f}초 | 속도: {tps:.2f} t/s")

# 통계 계산
avg_latency = sum(latencies) / 10
min_latency = min(latencies)
max_latency = max(latencies)
avg_tps = sum(tokens_per_second_list) / 10

# 기존 실험 양식과 100% 동일하게 출력
print("\n" + "="*40)
print(f"Report ({model_name})")
print(f"Avg Time: {avg_latency:.4f}s")
print(f"Min Time: {min_latency:.4f}s")
print(f"Max Time: {max_latency:.4f}s")
print(f"Avg Tokens/Sec: {avg_tps:.2f} t/s")
print(f"Model VRAM: {model_vram_mb:.2f} MB")
print(f"Peak VRAM: {model_vram_mb:.2f} MB")
print("="*40)

clear_vram(model_name)


Report (qwen2.5-coder:7b)
Avg Time: 6.0846s
Min Time: 4.8064s
Max Time: 13.2861s
Avg Tokens/Sec: 13.05 t/s
Model VRAM: 4395.67 MB
Peak VRAM: 4395.67 MB


##### Llama-3.1-8B_GGUF

In [ ]:
model_name = 'hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q4_K_M'

# 워밍업 (모델을 VRAM에 확실히 로드)
ollama.chat(model=model_name, messages=[{'role': 'user', 'content': 'hi'}], options={'num_ctx': 2048})

# Ollama ps API를 통해 VRAM 사용량 가져오기 (MB 단위 변환)
model_vram_mb = 0
get_vram(model_name)

latencies = []
tokens_per_second_list = []

for i in range(10):
    start_time = time.time()
    
    # options 파라미터를 사용해 max_new_tokens=64 와 동일한 효과 적용
    response = ollama.chat(model=model_name, messages=[
        {'role': 'user', 'content': 'AI의 장점은?'},
    ], options={
        'num_predict': 64  # 최대 생성 토큰 64개로 제한
    })
    
    end_time = time.time()
    iter_latency = end_time - start_time
    
    # Ollama가 자체적으로 계산한 순수 평가 시간 및 토큰 수 가져오기 (통신 딜레이 제외)
    eval_count = response.get('eval_count', 0) # 생성된 토큰 수
    eval_duration_ns = response.get('eval_duration', 0) # 생성에 걸린 나노초
    
    if eval_duration_ns > 0:
        tps = eval_count / (eval_duration_ns / 1e9) # 초당 토큰 생성 수 (TPS)
    else:
        tps = 0
        
    latencies.append(iter_latency)
    tokens_per_second_list.append(tps)
    
    # 진행 상황이 궁금하시면 아래 주석을 해제하세요.
    # print(f"[{i+1}/10] 소요 시간: {iter_latency:.2f}초 | 속도: {tps:.2f} t/s")

# 통계 계산
avg_latency = sum(latencies) / 10
min_latency = min(latencies)
max_latency = max(latencies)
avg_tps = sum(tokens_per_second_list) / 10

# 기존 실험 양식과 100% 동일하게 출력
print("\n" + "="*40)
print(f"Report ({model_name})")
print(f"Avg Time: {avg_latency:.4f}s")
print(f"Min Time: {min_latency:.4f}s")
print(f"Max Time: {max_latency:.4f}s")
print(f"Avg Tokens/Sec: {avg_tps:.2f} t/s")
print(f"Model VRAM: {model_vram_mb:.2f} MB")
print(f"Peak VRAM: {model_vram_mb:.2f} MB")
print("="*40)

clear_vram(model_name)


Report (hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q4_K_M)
Avg Time: 8.0233s
Min Time: 5.6339s
Max Time: 15.2123s
Avg Tokens/Sec: 8.92 t/s
Model VRAM: 4363.98 MB
Peak VRAM: 4363.98 MB


##### Dolphin3.0-Llama3.1-8B_GGUF

In [ ]:
model_name = 'hf.co/dphn/Dolphin3.0-Llama3.1-8B-GGUF:Q4_K_M'

# 워밍업 (모델을 VRAM에 확실히 로드)
ollama.chat(model=model_name, messages=[{'role': 'user', 'content': 'hi'}], options={'num_ctx': 2048})

# Ollama ps API를 통해 VRAM 사용량 가져오기 (MB 단위 변환)
model_vram_mb = 0
get_vram(model_name)

latencies = []
tokens_per_second_list = []

for i in range(10):
    start_time = time.time()
    
    # options 파라미터를 사용해 max_new_tokens=64 와 동일한 효과 적용
    response = ollama.chat(model=model_name, messages=[
        {'role': 'user', 'content': 'AI의 장점은?'},
    ], options={
        'num_predict': 64  # 최대 생성 토큰 64개로 제한
    })
    
    end_time = time.time()
    iter_latency = end_time - start_time
    
    # Ollama가 자체적으로 계산한 순수 평가 시간 및 토큰 수 가져오기 (통신 딜레이 제외)
    eval_count = response.get('eval_count', 0) # 생성된 토큰 수
    eval_duration_ns = response.get('eval_duration', 0) # 생성에 걸린 나노초
    
    if eval_duration_ns > 0:
        tps = eval_count / (eval_duration_ns / 1e9) # 초당 토큰 생성 수 (TPS)
    else:
        tps = 0
        
    latencies.append(iter_latency)
    tokens_per_second_list.append(tps)
    
    # 진행 상황이 궁금하시면 아래 주석을 해제하세요.
    # print(f"[{i+1}/10] 소요 시간: {iter_latency:.2f}초 | 속도: {tps:.2f} t/s")

# 통계 계산
avg_latency = sum(latencies) / 10
min_latency = min(latencies)
max_latency = max(latencies)
avg_tps = sum(tokens_per_second_list) / 10

# 기존 실험 양식과 100% 동일하게 출력
print("\n" + "="*40)
print(f"Report ({model_name})")
print(f"Avg Time: {avg_latency:.4f}s")
print(f"Min Time: {min_latency:.4f}s")
print(f"Max Time: {max_latency:.4f}s")
print(f"Avg Tokens/Sec: {avg_tps:.2f} t/s")
print(f"Model VRAM: {model_vram_mb:.2f} MB")
print(f"Peak VRAM: {model_vram_mb:.2f} MB")
print("="*40)

clear_vram(model_name)


Report (hf.co/dphn/Dolphin3.0-Llama3.1-8B-GGUF:Q4_K_M)
Avg Time: 8.1391s
Min Time: 7.2865s
Max Time: 11.8086s
Avg Tokens/Sec: 8.70 t/s
Model VRAM: 4363.99 MB
Peak VRAM: 4363.99 MB


##### gemma-2-2b-it-abliterated-GGUF

In [ ]:
model_name = 'hf.co/bartowski/gemma-2-2b-it-abliterated-GGUF:Q4_K_M'

# 워밍업 (모델을 VRAM에 확실히 로드)
ollama.chat(model=model_name, messages=[{'role': 'user', 'content': 'hi'}], options={'num_ctx': 2048})

# Ollama ps API를 통해 VRAM 사용량 가져오기 (MB 단위 변환)
model_vram_mb = 0
get_vram(model_name)

latencies = []
tokens_per_second_list = []

for i in range(10):
    start_time = time.time()
    
    # options 파라미터를 사용해 max_new_tokens=64 와 동일한 효과 적용
    response = ollama.chat(model=model_name, messages=[
        {'role': 'user', 'content': 'AI의 장점은?'},
    ], options={
        'num_predict': 64  # 최대 생성 토큰 64개로 제한
    })
    
    end_time = time.time()
    iter_latency = end_time - start_time
    
    # Ollama가 자체적으로 계산한 순수 평가 시간 및 토큰 수 가져오기 (통신 딜레이 제외)
    eval_count = response.get('eval_count', 0) # 생성된 토큰 수
    eval_duration_ns = response.get('eval_duration', 0) # 생성에 걸린 나노초
    
    if eval_duration_ns > 0:
        tps = eval_count / (eval_duration_ns / 1e9) # 초당 토큰 생성 수 (TPS)
    else:
        tps = 0
        
    latencies.append(iter_latency)
    tokens_per_second_list.append(tps)
    
    # 진행 상황이 궁금하시면 아래 주석을 해제하세요.
    # print(f"[{i+1}/10] 소요 시간: {iter_latency:.2f}초 | 속도: {tps:.2f} t/s")

# 통계 계산
avg_latency = sum(latencies) / 10
min_latency = min(latencies)
max_latency = max(latencies)
avg_tps = sum(tokens_per_second_list) / 10

# 기존 실험 양식과 100% 동일하게 출력
print("\n" + "="*40)
print(f"Report ({model_name})")
print(f"Avg Time: {avg_latency:.4f}s")
print(f"Min Time: {min_latency:.4f}s")
print(f"Max Time: {max_latency:.4f}s")
print(f"Avg Tokens/Sec: {avg_tps:.2f} t/s")
print(f"Model VRAM: {model_vram_mb:.2f} MB")
print(f"Peak VRAM: {model_vram_mb:.2f} MB")
print("="*40)

clear_vram(model_name)


Report (hf.co/bartowski/gemma-2-2b-it-abliterated-GGUF:Q4_K_M)
Avg Time: 1.1339s
Min Time: 0.8404s
Max Time: 3.6994s
Avg Tokens/Sec: 106.97 t/s
Model VRAM: 2336.17 MB
Peak VRAM: 2336.17 MB
VRAM 정리


In [6]:
model_name = 'hf.co/TeichAI/Qwen3-8B-Gemini-2.5-Flash-Distill-GGUF:Q4_K_M'

# 워밍업 (모델을 VRAM에 확실히 로드)
ollama.chat(model=model_name, messages=[{'role': 'user', 'content': 'hi'}], options={'num_ctx': 2048})

# Ollama ps API를 통해 VRAM 사용량 가져오기 (MB 단위 변환)
model_vram_mb = 0
get_vram(model_name)

latencies = []
tokens_per_second_list = []

for i in range(10):
    start_time = time.time()
    
    # options 파라미터를 사용해 max_new_tokens=64 와 동일한 효과 적용
    response = ollama.chat(model=model_name, messages=[
        {'role': 'user', 'content': 'AI의 장점은?'},
    ], options={
        'num_predict': 64  # 최대 생성 토큰 64개로 제한
    })
    
    end_time = time.time()
    iter_latency = end_time - start_time
    
    # Ollama가 자체적으로 계산한 순수 평가 시간 및 토큰 수 가져오기 (통신 딜레이 제외)
    eval_count = response.get('eval_count', 0) # 생성된 토큰 수
    eval_duration_ns = response.get('eval_duration', 0) # 생성에 걸린 나노초
    
    if eval_duration_ns > 0:
        tps = eval_count / (eval_duration_ns / 1e9) # 초당 토큰 생성 수 (TPS)
    else:
        tps = 0
        
    latencies.append(iter_latency)
    tokens_per_second_list.append(tps)
    
    # 진행 상황이 궁금하시면 아래 주석을 해제하세요.
    # print(f"[{i+1}/10] 소요 시간: {iter_latency:.2f}초 | 속도: {tps:.2f} t/s")

# 통계 계산
avg_latency = sum(latencies) / 10
min_latency = min(latencies)
max_latency = max(latencies)
avg_tps = sum(tokens_per_second_list) / 10

# 기존 실험 양식과 100% 동일하게 출력
print("\n" + "="*40)
print(f"Report ({model_name})")
print(f"Avg Time: {avg_latency:.4f}s")
print(f"Min Time: {min_latency:.4f}s")
print(f"Max Time: {max_latency:.4f}s")
print(f"Avg Tokens/Sec: {avg_tps:.2f} t/s")
print(f"Model VRAM: {model_vram_mb:.2f} MB")
print(f"Peak VRAM: {model_vram_mb:.2f} MB")
print("="*40)

clear_vram(model_name)


Report (hf.co/TeichAI/Qwen3-8B-Gemini-2.5-Flash-Distill-GGUF:Q4_K_M)
Avg Time: 9.9746s
Min Time: 6.9720s
Max Time: 33.0766s
Avg Tokens/Sec: 9.20 t/s
Model VRAM: 0.00 MB
Peak VRAM: 0.00 MB
VRAM 정리
